# Credit Card Fraud Detection — Exploratory Analysis

This notebook mirrors the EDA performed in the Streamlit dashboard and serves
as a standalone research artifact.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA

from src.utils import load_raw_data, fraud_stats, FEATURE_COLS, TARGET_COL

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 1. Load Data

In [ ]:
df = load_raw_data()
stats = fraud_stats(df)
print(f'Rows: {stats["total"]:,}  |  Fraud: {stats["fraud"]:,}  |  Fraud %: {stats["fraud_pct"]}%')
df.head()

## 2. Class Distribution

In [ ]:
fig = px.pie(
    names=['Legitimate', 'Fraud'],
    values=[stats['legit'], stats['fraud']],
    title='Class Distribution',
    hole=0.5,
    color_discrete_sequence=['#3B82F6', '#EF4444'],
)
fig.show()

## 3. Amount Distribution

In [ ]:
fig = px.histogram(
    df.sample(5000, random_state=42), x='Amount', color='Class',
    barmode='overlay', nbins=60,
    color_discrete_map={0: '#3B82F6', 1: '#EF4444'},
    title='Amount Distribution by Class',
)
fig.show()

## 4. Correlation Heatmap

In [ ]:
corr = df[FEATURE_COLS + [TARGET_COL]].corr()
fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr, ax=ax, cmap='RdBu', center=0, fmt='.1f', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. PCA Visualisation

In [ ]:
df_pca = df.sample(5000, random_state=42)
pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(df_pca[FEATURE_COLS].fillna(0))
df_pca = df_pca.copy()
df_pca['PC1'] = components[:, 0]
df_pca['PC2'] = components[:, 1]

fig = px.scatter(
    df_pca, x='PC1', y='PC2', color=df_pca[TARGET_COL].astype(str),
    color_discrete_map={'0': '#3B82F6', '1': '#EF4444'},
    opacity=0.5,
    title=f'PCA — {sum(pca.explained_variance_ratio_)*100:.1f}% variance explained',
)
fig.show()

## 6. Full Pipeline Run

In [ ]:
from src.preprocess import build_pipeline
from src.train import train_all
from src.evaluate import evaluate_all, metrics_dataframe

pipeline = build_pipeline(imbalance_method='SMOTE')
models = train_all(pipeline['X_train'], pipeline['y_train'])
metrics = evaluate_all(models, pipeline['X_test'], pipeline['y_test'], persist=False)
metrics_dataframe(metrics).sort_values('ROC AUC', ascending=False)